In [ ]:
from huggingface_hub import HfFileSystem
import pandas as pd

fs = HfFileSystem()
files = fs.glob("datasets/phreshphish/phreshphish/**/*.parquet")
print(len(files), "files")
files

In [ ]:
import time

train_files = [f for f in files if "/train-" in f]
test_files  = [f for f in files if "/test-" in f]
print(len(train_files), "train files,", len(test_files), "test files")

def load(file_list):
    parts = []
    for i, f in enumerate(file_list, start=1):
        t = time.time()
        df = pd.read_parquet(f"hf://{f}", columns=["url", "label", "date"])
        parts.append(df)
        print(f"{i}/{len(file_list)}  {f.split('/')[-1]}  {len(df):,} rows  {time.time()-t:.0f}s")
    return pd.concat(parts, ignore_index=True)

test = load(test_files)
train = load(train_files)
print(f"train: {len(train):,} rows   test: {len(test):,} rows")

In [ ]:
train.to_parquet("../../data/phresh_train.parquet", index=False)
test.to_parquet("../../data/phresh_test.parquet", index=False)

import os
for f in ["phresh_train.parquet", "phresh_test.parquet"]:
    size = os.path.getsize(f"../../data/{f}") / 1e6
    print(f, f"{size:.0f} MB")

In [ ]:
for df in (train, test):
    df["date"] = pd.to_datetime(df["date"])
    df["month"] = df["date"].dt.to_period("M")
    df["phish"] = (df["label"] == "phish").astype(int)
    df["url_len"] = df["url"].str.len()
    df["path_len"] = df["url"].str.replace(r"^\w+://[^/]+", "", regex=True).str.len()

for name, df in [("train", train), ("test", test)]:
    print(f"{name}: {len(df):,} rows | phishing share {df.phish.mean():.1%} | "
          f"dates {df.date.min().date()} to {df.date.max().date()}")

In [ ]:
monthly = train.groupby("month")["phish"].agg(share="mean", rows="count")
monthly["share"] = (monthly["share"] * 100).round(1)
print("Rows on Sep 8: train", (train.date.dt.date.astype(str) == "2025-09-08").sum(),
      "| test", (test.date.dt.date.astype(str) == "2025-09-08").sum())
monthly

In [ ]:
test_monthly = test.groupby("month")["phish"].agg(share="mean", rows="count")
test_monthly["share"] = (test_monthly["share"] * 100).round(1)
test_monthly

In [ ]:
import matplotlib.pyplot as plt

print(train.groupby("label")[["url_len", "path_len"]].describe().round(0).T)

fig, ax = plt.subplots(figsize=(8, 4))
for lbl in ["benign", "phish"]:
    train.loc[train.label == lbl, "path_len"].clip(upper=200).hist(
        bins=50, alpha=0.5, label=lbl, ax=ax, density=True)
ax.set_xlabel("path length (capped at 200)")
ax.set_ylabel("share of URLs")
ax.set_title("Path length by class (train)")
ax.legend()
plt.savefig("path_len.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
train["no_path"]  = train["path_len"] <= 1
train["over_1000"] = train["url_len"] > 1000
train["https"]    = train["url"].str.startswith("https")
train["ip_host"]  = train["url"].str.contains(r"^\w+://\d+\.\d+\.\d+\.\d+", regex=True)

for col in ["no_path", "over_1000", "https", "ip_host"]:
    t = pd.crosstab(train[col], train["label"])
    t["phish_%"] = (100 * t["phish"] / t.sum(axis=1)).round(1)
    print(f"--- {col} ---")
    print(t, "\n")